In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pauli 기저로 관측값 지정하기

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
```
</details>

양자역학에서 관측값(observable)은 측정 가능한 물리적 성질에 해당합니다.
예를 들어 스핀 시스템을 고려할 때, 시스템의 에너지를 측정하거나 자기화(magnetization) 또는 스핀 간의 상관관계와 같이 스핀의 정렬에 관한 정보를 얻고자 할 수 있습니다.

양자 컴퓨터에서 $n$-Qubit 관측값 $O$를 측정하려면, 이를 Pauli 연산자의 텐서 곱의 합으로 표현해야 합니다. 즉,

$$
O = \sum_{k=1}^K \alpha_k P_k,~~ P_k \in {I, X, Y, Z}^{\otimes n},~~ \alpha_k \in \mathbb{R},
$$

여기서

$$
I = \begin{pmatrix}
1 & 0 \\ 0 & 1
\end{pmatrix}
~~
X = \begin{pmatrix}
0 & 1 \\ 1 & 0
\end{pmatrix}
~~
Y = \begin{pmatrix}
0 & -i \\ i & 0
\end{pmatrix}
~~
Z = \begin{pmatrix}
1 & 0 \\ 0 & -1
\end{pmatrix}
$$

이고, 관측값이 에르미트(Hermitian)라는 사실, 즉 $O^\dagger = O$를 이용합니다. $O$가 에르미트가 아닌 경우에도 Pauli의 합으로 분해할 수 있지만, 계수 $\alpha_k$가 복소수가 됩니다.

많은 경우, 관측값은 관심 있는 시스템을 Qubit에 매핑한 후 자연스럽게 이 표현으로 지정됩니다.
예를 들어, 스핀-1/2 시스템은 이징 해밀토니안(Ising Hamiltonian)으로 매핑될 수 있습니다.

$$
H = \sum_{\langle i, j\rangle} Z_i Z_j - \sum_{i=1}^n X_i,
$$

여기서 인덱스 $\langle i, j\rangle$는 상호작용하는 스핀들을 나타내며, 스핀들은 $X$ 방향의 횡방향 자기장(transversal field)에 놓여 있습니다.
아래 첨자 인덱스는 어떤 Qubit에 Pauli 연산자가 작용하는지를 나타냅니다. 즉, $X_i$는 Qubit $i$에 $X$ 연산자를 적용하고 나머지는 변경하지 않습니다.

Qiskit SDK에서는 다음 코드로 이 해밀토니안을 구성할 수 있습니다.

In [1]:
from qiskit.quantum_info import SparsePauliOp

# define the number of qubits
n = 12

# define the single Pauli terms as ("Paulis", [indices], coefficient)
interactions = [
    ("ZZ", [i, i + 1], 1) for i in range(n - 1)
]  # we assume spins on a 1D line
field = [("X", [i], -1) for i in range(n)]

# build the operator
hamiltonian = SparsePauliOp.from_sparse_list(
    interactions + field, num_qubits=n
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIIIX', 'IIIIIIIIIIXI', 'IIIIIIIIIXII', 'IIIIIIIIXIII', 'IIIIIIIXIIII', 'IIIIIIXIIIII', 'IIIIIXIIIIII', 'IIIIXIIIIIII', 'IIIXIIIIIIII', 'IIXIIIIIIIII', 'IXIIIIIIIIII', 'XIIIIIIIIIII'],
              coeffs=[ 1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
 -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j])


If we would like to measure the energy the observable is the Hamiltonian itself. Alternatively, we could be
interested in measuring system properties like the average magnetization by counting the number of spins
aligned in the $Z$-direction with the observable

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

For observables that are not given in terms of Pauli operators but in a matrix form, we first have to reformulate them
in the Pauli basis in order to evaluate them on a quantum computer.
We are always able to find such a representation as the Pauli matrices form a basis for the Hermitian $2^n \times 2^n$ matrices.
We expand the observable $O$ as

$$
O = \sum_{P \in \{I, X, Y, Z\}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

where the sum runs over all possible $n$-qubit Pauli terms and $\mathrm{Tr}(\cdot)$ is the trace of a matrix, which acts as inner product.
You can implement this decomposition  from a matrix to Pauli terms using the `SparsePauliOp.from_operator` method, like so:

In [2]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

matrix = np.array(
    [[-1, 0, 0.5, -1], [0, 1, 1, 0.5], [0.5, 1, -1, 0], [-1, 0.5, 0, 1]]
)

observable = SparsePauliOp.from_operator(matrix)
print(observable)

SparsePauliOp(['IZ', 'XI', 'YY'],
              coeffs=[-1. +0.j,  0.5+0.j,  1. -0.j])


에너지를 측정하고자 한다면 관측값은 해밀토니안 자체가 됩니다. 또는 다음 관측값으로 $Z$ 방향으로 정렬된 스핀의 수를 세어 평균 자기화와 같은 시스템 속성을 측정할 수도 있습니다.

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

Pauli 연산자로 주어지지 않고 행렬 형태로 주어진 관측값의 경우, 양자 컴퓨터에서 평가하기 위해 먼저 Pauli 기저로 재표현해야 합니다.
Pauli 행렬이 에르미트 $2^n \times 2^n$ 행렬의 기저를 이루기 때문에, 우리는 항상 이러한 표현을 찾을 수 있습니다.
관측값 $O$를 다음과 같이 전개합니다.

$$
O = \sum_{P \in {I, X, Y, Z}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

여기서 합산은 가능한 모든 $n$-Qubit Pauli 항에 대해 이루어지며, $\mathrm{Tr}(\cdot)$는 내적으로 작용하는 행렬의 대각합(trace)입니다.
`SparsePauliOp.from_operator` 메서드를 사용하면 행렬에서 Pauli 항으로의 이 분해를 다음과 같이 구현할 수 있습니다.

In [3]:
from qiskit.circuit import QuantumCircuit

# create a circuit, where we would like to measure
# q0 in the X basis, q1 in the Y basis and q2 in the Z basis
circuit = QuantumCircuit(3)
circuit.ry(0.8, 0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.barrier()

# diagonalize X with the Hadamard gate
circuit.h(0)

# diagonalize Y with Hadamard as S^\dagger
circuit.sdg(1)
circuit.h(1)

# the Z basis is the default, no action required here

# measure all qubits
circuit.measure_all()
circuit.draw("mpl")

<Image src="../docs/images/guides/specify-observables-pauli/extracted-outputs/ce4b1984-ebe0-44f6-a78c-d67b9e9bb361-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
  -  Read the [SparsePauliOp API](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp#sparsepauliop) reference.
</Admonition>